# 05 - Agile modeling

## Descripcion

Proceso de entrenamiento del modelo final. Utiliza el flujo de trabajo agile para entrenar un modelo de clasificacion con perch_8 utilizando la train DB. 

In [ ]:
# Imports

import os

from matplotlib import pyplot as plt
import numpy as np

from perch_hoplite.agile import audio_loader
from perch_hoplite.agile import classifier
from perch_hoplite.agile import classifier_data
from perch_hoplite.agile import embedding_display
from perch_hoplite.agile import source_info
from perch_hoplite.db  import brutalism
from perch_hoplite.db import score_functions
from perch_hoplite.db  import search_results
from perch_hoplite.db import sqlite_usearch_impl
from perch_hoplite.zoo import model_configs
from perch_hoplite.zoo import taxonomy_model_tf

### Ruta a la carpeta que contiene la db TRAIN

In [ ]:
train_db_path = '/mnt/d/Taboga/Train/train_db' 

### Cargar database

In [ ]:
# @title Load model and connect to database {vertical-output: true}
# Identifier (e.g. name) to attach to labels produced during validation.
annotator_id = 'linnaeus'  # @param {type: 'string'}

db = sqlite_usearch_impl.SQLiteUSearchDB.create(train_db_path)
db_model_config = db.get_metadata('model_config')
embed_config = db.get_metadata('audio_sources')
model_class = model_configs.get_model_class(db_model_config.model_key)
embedding_model = model_class.from_config(db_model_config.model_config)
audio_sources = source_info.AudioSources.from_config_dict(embed_config)
if hasattr(embedding_model, 'window_size_s'):
  window_size_s = embedding_model.window_size_s
else:
  window_size_s = 5.0
audio_filepath_loader = audio_loader.make_filepath_loader(
    audio_sources=audio_sources,
    window_size_s=window_size_s,
    sample_rate_hz=embedding_model.sample_rate,
)


***Corregir problema de compatibilidad de usearch (devuelve tuple, y debe ser numpy*** 

In [ ]:
import numpy as np
from perch_hoplite.db import sqlite_usearch_impl

def get_embeddings_batch_fixed(self, window_ids):
    embeddings_batch = self.ui.get(window_ids)

    if isinstance(embeddings_batch, tuple):
        embeddings_batch = np.stack(embeddings_batch)

    if not isinstance(embeddings_batch, np.ndarray):
        raise RuntimeError(
            f"Expected np.ndarray or tuple, got {type(embeddings_batch)}"
        )

    return embeddings_batch

sqlite_usearch_impl.SQLiteUSearchDB.get_embeddings_batch = get_embeddings_batch_fixed

ids = brutalism.get_brute_search_ids(db, sample_size=10, rng_seed=42)
emb = db.get_embeddings_batch(ids)

print(type(emb))
print(emb.shape)

# Busqueda inicial

### Cargar la plantilla para consulta (query)
El `query_uri` puede ser una URL, una ruta de archivo o un ID de Xeno-Canto (como `xc105133`, que contiene un zorzal de bosque (`woothr`)). Para mejores resultados en el entrenamiento final, **se recomienda utilizar una plantilla tomada de los propios datos** de manera que sea lo mas similar a las señal esperada. Esta plantilla debe ser de la mejor calidad posible.


In [ ]:
query_uri = '/mnt/d/Taboga/Cebus01.wav'  # Ruta al archivo que se desea utilizar como plantilla
query_label = 'cebimi'  # Nombre que se le dara a la etiqueta de anotacion

query = embedding_display.QueryDisplay(
    uri=query_uri, offset_s=0.0, window_size_s=5.0, sample_rate_hz=32000)
_ = query.display_interactive()

## Entrenamiento - inicio de loop

In [ ]:
# Embed the Query and Search

# numero de resultados a mostrar
num_results = 50  # @param
query_embedding = embedding_model.embed(
    query.get_audio_window()).embeddings[0, 0]

# @markdown If checked, search for examples near a particular target score.
target_sampling = False  # @param {type: 'boolean'}

# @markdown When target sampling, target this score.
target_score = -1.0  # @param
if not target_sampling:
  target_score = None

# @markdown If True, search the full DB. Otherwise, use approximate
# @markdown nearest-neighbor search.
exact_search = False  # @param {type: 'boolean'}

results = db.search(
    query_embedding,
    search_list_size=num_results,
    approximate=exact_search,
    target_score=target_score,
)
# Get a random batch of scores to plot the score distribution.
scores = brutalism.get_random_embedding_scores(
    db, query_embedding, score_fn=score_functions.get_score_fn('dot'),
    sample_size=2_048,
    rng_seed=42,
)
_ = plt.hist(scores, bins=25, density=True, alpha=0.5)
hit_scores = [r.sort_score for r in results.search_results]
plt.scatter(hit_scores, np.zeros_like(hit_scores), marker='|',
            color='r', alpha=0.5)


In [ ]:
# Display Results
display_results = embedding_display.EmbeddingDisplayGroup.from_search_results(
    results,
    db,
    sample_rate_hz=32000,
    frame_rate=100,
    audio_loader=audio_filepath_loader,
)
display_results.display(positive_labels=[query_label],  paged_mode = False)

In [ ]:
# @title Save data labels {vertical-output: true}

print("Annotations before saving new labels:", len(db.get_all_annotations()))

db.insert_annotations(
    display_results.harvest_labels(annotator_id),
    handle_duplicates="skip",
)

print("Annotations after saving new labels:", len(db.get_all_annotations()))

# Crear modelo clasificador inicial

Nombre del modelo inicial que se guardara

In [ ]:
initial_model_name = 'agile_classifier_01.pt'

In [ ]:
#Set of labels to classify. If None, auto-populated from the DB.
target_labels = None  

# Classifier traning hyperparams. These should not require tuning.
learning_rate = 1e-3 
weak_neg_weight = 0.05  
l2_mu = 0.000  
num_steps = 128  

train_ratio = 0.9  
batch_size = 128  
weak_negatives_batch_size = 128  
loss_fn_name = 'bce'  

data_manager = classifier_data.AgileDataManager(
    target_labels=target_labels,
    db=db,
    train_ratio=train_ratio,
    min_eval_examples=1,
    batch_size=batch_size,
    weak_negatives_batch_size=weak_negatives_batch_size,
    rng=np.random.default_rng(seed=5))
print('Training for target labels : ')
print(data_manager.get_target_labels())
linear_classifier, eval_scores = classifier.train_linear_classifier(
    data_manager=data_manager,
    learning_rate=learning_rate,
    weak_neg_weight=weak_neg_weight,
    num_train_steps=num_steps,
)
print('\n' + '-' * 80)
top1 = eval_scores['top1_acc']
print(f'top-1      {top1:.3f}')
rocauc = eval_scores['roc_auc']
print(f'roc_auc    {rocauc:.3f}')
cmap = eval_scores['cmap']
print(f'cmap       {cmap:.3f}')

# Save linear classifier.
linear_classifier.save(os.path.join(train_db_path, initial_model_name))

print(f'Modelo guardado en {os.path.join(train_db_path, initial_model_name)}')

## Inicio del loop de entrenamiento

Actualizar el modelo que se va a utilizar en cada iteracion

Por ejemplo en la primera iteracion:  
  model_path = "/mnt/d/Taboga/Train/train_db/agile_classifier_01.pt"  
  model_output_path = "/mnt/d/Taboga/Train/train_db/agile_classifier_02.pt"

En la segunda iteracion:  
  model_path = "/mnt/d/Taboga/Train/train_db/agile_classifier_02.pt"  
  model_output_path = "/mnt/d/Taboga/Train/train_db/agile_classifier_03.pt"

In [ ]:
# Nombre del modelo a utilizar en esta iteracion
model_path = "/mnt/d/Taboga/Train/train_db/agile_classifier_02.pt"

# Nombre con el que se guardara el nuevo modelo luego de entrenar
model_output_path = "/mnt/d/Taboga/Train/train_db/agile_classifier_03.pt"

In [ ]:
# --- 1. Cargar modelo lineal ---
custom_classifier = classifier.LinearClassifier.load(model_path)

for name, value in vars(custom_classifier).items():
    if any(word in name.lower() for word in ("class", "target")):
        print(f"Clasificador cargado. Este clasificador contiene las siguientes anotaciones: \n {name}: {value}")
        c_label = value


In [ ]:
# Review Classifier Results {vertical-output: true}

target_label = 'cebimi'  
num_results = 50  

#Set of labels to classify. If None, auto-populated from the DB.
target_labels = None  

# Classifier traning hyperparams. These should not require tuning.
learning_rate = 1e-3 
weak_neg_weight = 0.05  
l2_mu = 0.000  
num_steps = 128  

train_ratio = 0.9  
batch_size = 128  
weak_negatives_batch_size = 128  
loss_fn_name = 'bce'  

data_manager = classifier_data.AgileDataManager(
    target_labels=target_labels,
    db=db,
    train_ratio=train_ratio,
    min_eval_examples=1,
    batch_size=batch_size,
    weak_negatives_batch_size=weak_negatives_batch_size,
    rng=np.random.default_rng(seed=5))
print('Training for target labels : ')
print(data_manager.get_target_labels())

target_label_idx = data_manager.get_target_labels().index(target_label)
class_query = custom_classifier.beta[:, target_label_idx]
bias = custom_classifier.beta_bias[target_label_idx]

#Number of (randomly selected) database entries to search over.
sample_size = 1_000_000  

# Whether to use margin-sampling. If checked, search for examples
# with logits near a particular target score (usually 0).
margin_sampling = False  

# When margin sampling, target this logit.
margin_target_score = -0.0  # @param
if not margin_sampling:
  margin_target_score = None
score_fn = score_functions.get_score_fn(
    'dot', bias=bias, target_score=margin_target_score)
results = brutalism.threaded_brute_search(
    db, class_query, num_results, score_fn=score_fn,
    sample_size=sample_size)

# Get a random batch of scores to plot the score distribution.
scores = brutalism.get_random_embedding_scores(
    db,
    class_query,
    score_fn=score_functions.get_score_fn('dot', bias=bias),
    sample_size=2_048,
    rng_seed=42,
)
plt.hist(scores, bins=25, density=True, alpha=0.5)
hit_scores = [r.sort_score for r in results.search_results]
_ = plt.scatter(hit_scores, np.zeros_like(hit_scores), marker='|',
            color='r', alpha=0.5)


In [ ]:
display_results = embedding_display.EmbeddingDisplayGroup.from_search_results(
    results,
    db,
    sample_rate_hz=32000,
    frame_rate=100,
    audio_loader=audio_filepath_loader,
)
display_results.display(positive_labels=[target_label],  paged_mode = False)

In [ ]:
# @title Save data labels {vertical-output: true}

print("Annotations before saving new labels:", len(db.get_all_annotations()))

db.insert_annotations(
    display_results.harvest_labels(annotator_id),
    handle_duplicates="skip",
)

print("Annotations after saving new labels:", len(db.get_all_annotations()))

In [ ]:
from perch_hoplite.db import datatypes

pos_ids = set(
    map(
        int,
        data_manager._get_window_ids_for_label_type(
            target_label,
            datatypes.LabelType.POSITIVE,
        ),
    )
)

neg_ids = set(
    map(
        int,
        data_manager._get_window_ids_for_label_type(
            target_label,
            datatypes.LabelType.NEGATIVE,
        ),
    )
)

conflicting_ids = sorted(pos_ids & neg_ids)

print("Ventanas positivas:", len(pos_ids))
print("Ventanas negativas:", len(neg_ids))
print("Ventanas con conflicto:", len(conflicting_ids))
print("Primeros conflictos:", conflicting_ids[:20])

In [ ]:
# @title Classifier training {vertical-output: true}

# @markdown Set of labels to classify. If None, auto-populated from the DB.
target_labels = None  # @param

# @markdown Classifier traning hyperparams. These should not require tuning.
learning_rate = 1e-3  # @param
weak_neg_weight = 0.05  # @param
l2_mu = 0.000  # @param
num_steps = 128  # @param

train_ratio = 0.9  # @param
batch_size = 128  # @param
weak_negatives_batch_size = 128  # @param
loss_fn_name = 'bce'  # @param ['hinge', 'bce']

data_manager = classifier_data.AgileDataManager(
    target_labels=target_labels,
    db=db,
    train_ratio=train_ratio,
    min_eval_examples=1,
    batch_size=batch_size,
    weak_negatives_batch_size=weak_negatives_batch_size,
    rng=np.random.default_rng(seed=5))
print('Training for target labels : ')
print(data_manager.get_target_labels())
linear_classifier, eval_scores = classifier.train_linear_classifier(
    data_manager=data_manager,
    learning_rate=learning_rate,
    weak_neg_weight=weak_neg_weight,
    num_train_steps=num_steps,
)


from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)
import pandas as pd
import numpy as np

# Etiqueta para la que se seleccionará el threshold
target_label = "cebimi"

target_label_idx = (
    data_manager.get_target_labels().index(target_label)
)

# Ground truth y logits del subconjunto de evaluación
y_true_raw = eval_scores["eval_labels"][:, target_label_idx]
y_score = eval_scores["eval_preds"][:, target_label_idx]

print("Valores presentes en eval_labels:", np.unique(y_true_raw))

# No continuar si existen etiquetas conflictivas o fraccionarias
if not np.all(np.isin(y_true_raw, [0.0, 1.0])):
    raise ValueError(
        "Existen ventanas con etiquetas conflictivas o fraccionarias. "
        "Revise las anotaciones positivas y negativas antes de "
        "seleccionar el threshold."
    )

y_true = y_true_raw.astype(int)

# Curva Precision–Recall
precision, recall, thresholds = precision_recall_curve(
    y_true,
    y_score,
)

# precision y recall tienen un elemento adicional
precision_t = precision[:-1]
recall_t = recall[:-1]

f1 = np.divide(
    2 * precision_t * recall_t,
    precision_t + recall_t,
    out=np.zeros_like(precision_t),
    where=(precision_t + recall_t) > 0,
)

threshold_results = pd.DataFrame({
    "threshold_logit": thresholds,
    "precision": precision_t,
    "recall": recall_t,
    "f1": f1,
})

best_idx = threshold_results["f1"].idxmax()
best = threshold_results.loc[best_idx]

recommended_threshold = float(best["threshold_logit"])

print("\nThreshold recomendado por máximo F1")
print(best)

# Matriz de confusión con el threshold seleccionado
y_pred = (y_score >= recommended_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1],
).ravel()

print("\nResultados en validación")
print(f"TP: {tp}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TN: {tn}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")
print(f"MCC:       {matthews_corrcoef(y_true, y_pred):.3f}")


In [ ]:
# Save linear classifier.
linear_classifier.save(model_output_path)
print(f"\nModelo guardado en {model_output_path}")

## Fin del loop de entrenamiento iterar nuevamente o cerrar la DB

## Cerrar la DB

In [ ]:
import gc


def close_hoplite_db(
    hoplite_db,
    commit=True,
):
    """
    Cierra una base Hoplite abierta.

    commit=True:
        Guarda cambios de SQLite y USearch antes de cerrar.

    commit=False:
        Descarta cambios SQLite pendientes, útil para bases
        abiertas solo para lectura o inferencia.
    """

    if hoplite_db is None:
        return

    try:
        if commit:
            hoplite_db.commit()
        else:
            try:
                hoplite_db.db.rollback()
            except Exception:
                pass

    finally:
        # Cerrar conexión SQLite
        try:
            hoplite_db.db.close()
        except Exception:
            pass

        # Liberar referencia al índice USearch
        try:
            hoplite_db.ui = None
        except Exception:
            pass

        gc.collect()

In [ ]:
close_hoplite_db(
    db,
    commit=True,
)

db = None

gc.collect()